# 🧪 Sesión 20 · Hito 2: API con Inyección de Dependencias, Auth JWT y RBAC

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿La seguridad de tu API resiste una auditoría?

---

## 🖥️ Así se verá el resultado final

```http
POST /api/registro   {"usuario":"ana","password":"secreta"}  -> 201
POST /api/token      {"usuario":"ana","password":"secreta"}  -> 200 + token
GET  /api/perfil     Bearer <token_user>                     -> 200
GET  /api/admin/panel Bearer <token_user>                    -> 403
POST /api/token      {"usuario":"admin","password":"admin123"} -> 200 + token
GET  /api/admin/panel Bearer <token_admin>                   -> 200
GET  /api/perfil     sin token                               -> 401

RESUMEN: 7/7 casos correctos. ¡Hito 2 listo!
```

_La seguridad no es una capa al final: es una propiedad de todo el flujo._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Integrar registro, login, token, perfil y panel administrativo.
- Verificar 401, 403 y 200 en el mismo flujo.
- Demostrar que las contraseñas nunca se filtran.

## ✅ Criterios de aceptación

El verificador ejecutará estos casos. Cada uno debe quedar en verde:

- [ ] Registro sin filtrar contraseñas → HTTP 201
- [ ] Login del usuario registrado → HTTP 200
- [ ] El usuario accede a su perfil → HTTP 200
- [ ] El usuario no entra al panel admin → HTTP 403
- [ ] El administrador obtiene su token → HTTP 200
- [ ] El administrador entra al panel → HTTP 200
- [ ] Sin token no hay acceso → HTTP 401

## 🧩 El reto

Integra el pipeline de seguridad completo del Bloque 2 en una API de libros con usuarios, roles y rutas protegidas.

**Requisitos del contrato**

- El administrador precargado es admin/admin123; el usuario ana se registra en el flujo.
- Registro de usuarios con contraseña hasheada (nunca en texto plano).
- Login que emite un JWT con el rol del usuario.
- Perfil protegido con Bearer para cualquier usuario autenticado.
- Panel administrativo solo para rol admin; usuario normal recibe 403.
- Sin token se responde 401 y las contraseñas nunca aparecen en las respuestas.

> **Pista conceptual:** Ordena el flujo como lo vive el cliente: registro, login, token, perfil, panel. Cada paso debe poder probarse de forma independiente.

## 📦 Entregable

Cuaderno en verde, bitácora `auditorias/auditoria_sesion20.md` y tag `v0.2-hito2` en tu repositorio.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [1]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [4]:
%%writefile app.py
from fastapi import FastAPI, HTTPException, Depends
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
from pydantic import BaseModel, Field
from passlib.context import CryptContext
from jose import jwt, JWTError
from datetime import datetime, timedelta, timezone

app = FastAPI(title="API Hito 2 - Seguridad")

# -----------------------------
# 1. Configuración de seguridad
# -----------------------------
SECRET_KEY = "clave-laboratorio-sesion20-cambiar"
ALGORITHM = "HS256"
ACCESS_TOKEN_EXPIRE_MINUTES = 30

pwd_context = CryptContext(
    schemes=["pbkdf2_sha256"],
    deprecated="auto"
)

bearer_scheme = HTTPBearer(auto_error=False)

# -----------------------------
# 2. Modelos de datos
# -----------------------------
class UsuarioEntrada(BaseModel):
    usuario: str = Field(min_length=3, max_length=50)
    password: str = Field(min_length=6, max_length=100)

class UsuarioRespuesta(BaseModel):
    usuario: str
    rol: str

class TokenRespuesta(BaseModel):
    access_token: str
    token_type: str

# -----------------------------
# 3. Almacenamiento en memoria
# -----------------------------
usuarios = {}

# Administrador precargado
usuarios["admin"] = {
    "password_hash": pwd_context.hash("admin123"),
    "rol": "admin"
}

# -----------------------------
# 4. Funciones de seguridad
# -----------------------------
def verificar_password(password, password_hash):
    return pwd_context.verify(password, password_hash)

def crear_token(usuario, rol):
    datos = {
        "sub": usuario,
        "rol": rol,
        "exp": datetime.now(timezone.utc)
               + timedelta(minutes=ACCESS_TOKEN_EXPIRE_MINUTES)
    }

    return jwt.encode(datos, SECRET_KEY, algorithm=ALGORITHM)

# -----------------------------
# 5. Dependencia: usuario autenticado
# -----------------------------
def obtener_usuario_actual(
    credenciales: HTTPAuthorizationCredentials = Depends(bearer_scheme)
):
    if credenciales is None:
        raise HTTPException(
            status_code=401,
            detail="Token requerido",
            headers={"WWW-Authenticate": "Bearer"}
        )

    try:
        payload = jwt.decode(
            credenciales.credentials,
            SECRET_KEY,
            algorithms=[ALGORITHM]
        )

        nombre_usuario = payload.get("sub")
        rol = payload.get("rol")

        if not nombre_usuario or not rol:
            raise HTTPException(
                status_code=401,
                detail="Token inválido",
                headers={"WWW-Authenticate": "Bearer"}
            )

        usuario = usuarios.get(nombre_usuario)

        if usuario is None or usuario["rol"] != rol:
            raise HTTPException(
                status_code=401,
                detail="Token inválido",
                headers={"WWW-Authenticate": "Bearer"}
            )

        return {
            "usuario": nombre_usuario,
            "rol": rol
        }

    except JWTError:
        raise HTTPException(
            status_code=401,
            detail="Token inválido o expirado",
            headers={"WWW-Authenticate": "Bearer"}
        )

# -----------------------------
# 6. Dependencia: solo administrador
# -----------------------------
def obtener_admin(
    usuario_actual: dict = Depends(obtener_usuario_actual)
):
    if usuario_actual["rol"] != "admin":
        raise HTTPException(
            status_code=403,
            detail="No tienes permisos de administrador"
        )

    return usuario_actual

# -----------------------------
# 7. Registro
# -----------------------------
@app.post("/api/registro", status_code=201)
def registrar_usuario(datos: UsuarioEntrada):
    if datos.usuario in usuarios:
        raise HTTPException(
            status_code=409,
            detail="El usuario ya existe"
        )

    usuarios[datos.usuario] = {
        "password_hash": pwd_context.hash(datos.password),
        "rol": "user"
    }

    return {
        "usuario": datos.usuario,
        "mensaje": "Usuario registrado"
    }

# -----------------------------
# 8. Login y generación de JWT
# -----------------------------
@app.post("/api/token", response_model=TokenRespuesta)
def login(datos: UsuarioEntrada):
    usuario = usuarios.get(datos.usuario)

    if usuario is None or not verificar_password(
        datos.password,
        usuario["password_hash"]
    ):
        raise HTTPException(
            status_code=401,
            detail="Usuario o contraseña incorrectos"
        )

    token = crear_token(datos.usuario, usuario["rol"])

    return {
        "access_token": token,
        "token_type": "bearer"
    }

# -----------------------------
# 9. Perfil protegido
# -----------------------------
@app.get("/api/perfil")
def perfil(
    usuario_actual: dict = Depends(obtener_usuario_actual)
):
    return {
        "usuario": usuario_actual["usuario"],
        "rol": usuario_actual["rol"]
    }

# -----------------------------
# 10. Panel administrativo
# -----------------------------
@app.get("/api/admin/panel")
def panel_admin(
    administrador: dict = Depends(obtener_admin)
):
    return {
        "mensaje": "Acceso autorizado al panel administrativo",
        "usuario": administrador["usuario"],
        "rol": administrador["rol"]
    }


Overwriting app.py


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Utilizar Python, FastAPI y Google Colab para desarrollar una API segura con JWT y RBAC. |
| **Rol** | La IA será un asistente técnico que me guiará en la implementación y revisión de la seguridad de mi API. |
| **Especificación** | Implementar registro, login, perfil protegido y panel de administrador, utilizando contraseñas hasheadas, tokens JWT y roles, con respuestas HTTP 200, 201, 401 y 403. |
| **Output** | Obtener app.py funcional, 7/7 pruebas en verde, bitácora de auditoría, cuaderno ejecutado y tag v0.2-hito2. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [5]:
from verificar import probar_http

casos = [{'descripcion': 'Registro sin filtrar contraseñas',
  'metodo': 'POST',
  'ruta': '/api/registro',
  'json': {'usuario': 'ana', 'password': 'secreta'},
  'estado': 201,
  'no_contiene': ['password', 'hash']},
 {'descripcion': 'Login del usuario registrado',
  'metodo': 'POST',
  'ruta': '/api/token',
  'json': {'usuario': 'ana', 'password': 'secreta'},
  'estado': 200,
  'guardar': {'token_user': 'access_token'}},
 {'descripcion': 'El usuario accede a su perfil',
  'metodo': 'GET',
  'ruta': '/api/perfil',
  'headers': {'Authorization': 'Bearer {token_user}'},
  'estado': 200,
  'json_contiene': {'usuario': 'ana'}},
 {'descripcion': 'El usuario no entra al panel admin',
  'metodo': 'GET',
  'ruta': '/api/admin/panel',
  'headers': {'Authorization': 'Bearer {token_user}'},
  'estado': 403},
 {'descripcion': 'El administrador obtiene su token',
  'metodo': 'POST',
  'ruta': '/api/token',
  'json': {'usuario': 'admin', 'password': 'admin123'},
  'estado': 200,
  'guardar': {'token_admin': 'access_token'}},
 {'descripcion': 'El administrador entra al panel',
  'metodo': 'GET',
  'ruta': '/api/admin/panel',
  'headers': {'Authorization': 'Bearer {token_admin}'},
  'estado': 200},
 {'descripcion': 'Sin token no hay acceso', 'metodo': 'GET', 'ruta': '/api/perfil', 'estado': 401}]

probar_http(casos, archivo="app.py", limpiar=[], env={})


VERIFICACIÓN DEL RESULTADO · 7 casos · app.py
✅ 01. Registro sin filtrar contraseñas
✅ 02. Login del usuario registrado
✅ 03. El usuario accede a su perfil
✅ 04. El usuario no entra al panel admin
✅ 05. El administrador obtiene su token
✅ 06. El administrador entra al panel
✅ 07. Sin token no hay acceso
------------------------------------------------------------------------------
RESULTADO: 7/7 casos correctos. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion20.md` a `auditorias/auditoria_sesion20.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué? R.- El resultado que más me costó fue integrar la autenticación con los permisos de los usuarios, porque tuve que entender cómo se genera y valida el token JWT y cómo se comprueba el rol para permitir o bloquear el acceso a las rutas.

   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato? R.- Decidí separar las funciones de autenticación y autorización para que una verificara si el usuario tenía un token válido y la otra comprobara si tenía permisos de administrador. También decidí utilizar almacenamiento en memoria para realizar las pruebas de esta actividad.

   - ¿Qué error detectó la verificación que tú no habías previsto? R.- La verificación me ayudó a comprobar que no bastaba con que el usuario iniciara sesión correctamente, sino que también debía validar los permisos antes de entrar al panel administrativo. Esto me permitió identificar la diferencia entre estar autenticado y estar autorizado.
   
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
